<a href="https://colab.research.google.com/github/gssmiles/flutter_ios_1/blob/main/Bahaa_AI_Dataset_Maker_For_RVC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p>&nbsp;👇 Subscribe To My Channel For More AI Updates And Tutorials 👇</p><p><a href="https://www.youtube.com/@BahaaAI" target="_blank">@BahaaAI</a></p><div class="separator" style="clear: both; text-align: center;"><a href="https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEib466daRFdGSKLhVpThmDsCccsGzdFE7-kWKcdN9I7tkgTZTq6uLIdJkDm-HQXSE3EXGW35cQj4r7__KOAcI9idy7Y-Cltg6kRmTQtYoaRhxPkj4r3BFP3llMx42admhdbnnKeU5-uW4y7-3D35Qhfshhtcb8XQNtZx8m_dWwzo3R0THNbZAxaJj9aUf4/s500/profile.jpg" style="margin-left: 1em; margin-right: 1em;"><img border="0" data-original-height="500" data-original-width="500" height="200" src="https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEib466daRFdGSKLhVpThmDsCccsGzdFE7-kWKcdN9I7tkgTZTq6uLIdJkDm-HQXSE3EXGW35cQj4r7__KOAcI9idy7Y-Cltg6kRmTQtYoaRhxPkj4r3BFP3llMx42admhdbnnKeU5-uW4y7-3D35Qhfshhtcb8XQNtZx8m_dWwzo3R0THNbZAxaJj9aUf4/w200-h200/profile.jpg" width="200" /></a></div>

---

<p>&nbsp;If you see this message, Click Cancel</p><div class="separator" style="clear: both; text-align: center;"><a imageanchor="1" style="margin-left: 1em; margin-right: 1em;"><img border="0" data-original-height="257" data-original-width="528" src="https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEgQ03JKlBn7e25COBDqSi1Ak8Ftxxr1xbQ04dPUMqLzIX_nYjsHZitBf8TLji_rTUZYZnSqBHE_i5G195T8N7kl8ik32tq8USlEZ9jvVOzE3jFCKXxx2EYF6rKmM5wLsMLEwTrRU9ms_5FnewU3T3XB7mOl2hsJmxGMrUNRdHAwPNB5LupYg9UnOtZFGHEE/s16000/Screenshot%202024-07-05%20121525.png" /></a></div>

In [ ]:
!pip uninstall -y tensorboard tb-nightly tensorflow tensorboardX
!pip install tensorboard==2.16.2

In [ ]:
#@title Step 1 : Installation

!pip install pydub
!apt-get install ffmpeg
# Install demucs
!pip install demucs

# Install gdown if not already installed
!pip install gdown

# Download the file from Google Drive
file_id = "1T6VfM21zaPZ_6fK3BSMgtn1fQwOyHR-g"
!gdown --id {file_id}

%run 3_Dataset_Maker_For_RVC.py

In [ ]:
#@title Step 2: Upload Audio Files (.mp3 or .wav Format)

import os
import shutil
from google.colab import files
from pydub import AudioSegment

# Function to clear and recreate a directory
def clear_directory(directory):
    if os.path.exists(directory):
        shutil.rmtree(directory)
    os.makedirs(directory, exist_ok=True)

# Clear and recreate the directories
clear_directory('uploads')
clear_directory('merged')

# Upload multiple audio files to the "uploads" directory
uploaded = files.upload()

# Save the uploaded files to the "uploads" directory
for filename in uploaded.keys():
    # Determine the file extension
    file_ext = os.path.splitext(filename)[1].lower()

    # Ensure the file is either MP3 or WAV
    if file_ext == '.mp3' or file_ext == '.wav':
        with open(os.path.join('uploads', filename), 'wb') as f:
            f.write(uploaded[filename])
    else:
        print(f"Ignoring file {filename} as it is not in .mp3 or .wav format.")

# Merge the uploaded audio files into one MP3 file
merged_audio = AudioSegment.empty()

# Iterate over each uploaded file and append it to the merged_audio
for filename in os.listdir('uploads'):
    file_ext = os.path.splitext(filename)[1].lower()
    if file_ext == '.mp3' or file_ext == '.wav':  # Ensure the file is MP3 or WAV
        audio = AudioSegment.from_file(os.path.join('uploads', filename))
        merged_audio += audio

# Export the merged audio to the "merged" directory as MP3
output_path = os.path.join('merged', 'merged_audio.mp3')
merged_audio.export(output_path, format='mp3')

print(f"Merged audio saved to {output_path}")


In [ ]:
#@title Step 3 : Create Dataset

# Demucs

# Function to clear directory
def clear_directory(directory):
    if os.path.exists(directory):
        shutil.rmtree(directory)
    os.makedirs(directory, exist_ok=True)

# Create and clear "vocals" directory
clear_directory('vocals')

# Define the model to use
demucs_model = "htdemucs"

# Path to the merged audio file
merged_audio_path = '/content/merged/merged_audio.mp3'

# Use demucs to separate the vocals from the merged audio file
!python3 -m demucs.separate -n {demucs_model} -o vocals --two-stems=vocals {merged_audio_path}

print("Vocals extracted and saved in the 'vocals' directory.")


# Remove Silence

# Define file paths
wav_file_path = '/content/vocals/vocals_cut.wav'
mp3_file_path = '/content/vocals/vocals_cut.mp3'

# Delete the files if they exist
if os.path.exists(wav_file_path):
    os.remove(wav_file_path)
    print(f"Deleted file: {wav_file_path}")

if os.path.exists(mp3_file_path):
    os.remove(mp3_file_path)
    print(f"Deleted file: {mp3_file_path}")


# Define paths and parameters
input_vocals_path = '/content/vocals/htdemucs/merged_audio/vocals.wav'
output_vocals_path = '/content/vocals/vocals_cut.wav'
silence_thresh = -40  # silence threshold (in dB)
min_silence_len = 10  # minimum length of silence (in ms) to be considered

# Load the vocals audio file
vocals_audio = AudioSegment.from_wav(input_vocals_path)

# Detect silence moments
silence_ranges = detect_silence(vocals_audio, min_silence_len=min_silence_len, silence_thresh=silence_thresh)

# Invert silence ranges to non-silence ranges
non_silence_ranges = []
start = 0
for start_silence, end_silence in silence_ranges:
    if start < start_silence:
        non_silence_ranges.append((start, start_silence))
    start = end_silence
if start < len(vocals_audio):
    non_silence_ranges.append((start, len(vocals_audio)))

# Concatenate non-silence audio segments
vocals_cut = AudioSegment.empty()
for start, end in non_silence_ranges:
    vocals_cut += vocals_audio[start:end]

# Export the result
vocals_cut.export(output_vocals_path, format='wav')

print(f"Silence removed and result saved to {output_vocals_path}")

In [ ]:
#@title Step 4 : Download Dataset File

# Step 1: Install necessary libraries
!pip install pydub

# Step 2: Convert WAV to MP3
from pydub import AudioSegment

# Load your WAV file
audio = AudioSegment.from_wav("/content/vocals/vocals_cut.wav")

# Export as MP3
audio.export("/content/dataset.mp3", format="mp3")

# Step 3: Download the file
from google.colab import files
files.download("/content/dataset.mp3")
